# Quickstart из исходного архива

Этот ноутбук показывает формат данных и собирает самый простой ответ в `output/quickstart_submission.csv`. Он нужен как пример запуска, его качество близко к константе

Если хочешь пройти весь путь решения, начни с `01_eda.ipynb`. Выбор итоговой модели и сравнение основных результатов лежат в `13_full_sequence.ipynb`

In [ ]:
# Открой ноутбук из корня проекта или из папки notebooks.
from pathlib import Path
import os
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if not (project_root / "data" / "train.csv").is_file():
    raise FileNotFoundError("Не найдена папка data. Открой ноутбук из проекта.")
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

In [ ]:
events.head()

## Смотрим на данные

Перед агрегацией проверим типы событий, значения `platform` и `user_agent`, пропуски и уникальность `cookie_id`

In [ ]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

## Суточное окно

Для каждой куки берём только события с `window_start_ts <= event_ts < window_end_ts`. Событие ровно на границе конца окна уже не входит в признаки

In [ ]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

## Простейшие признаки

Считаем число событий и число разных объявлений. Этого хватает для проверки пайплайна, но не для сильной модели

In [ ]:
def basic_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)

Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values
Xtr.head()

## Валидация

Тестовые дни идут после обучающих, поэтому проверяем модель на более поздних днях, не перемешивая куки случайным сплитом

Метрику берём из `utils/metric.py`. Одинаковые `score` проверяющая система обрабатывает одной группой: их нельзя делить порогом на части

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from utils.metric import precision_at_recall   # официальная реализация, ей же считает автопроверка

is_valid = train.window_start_ts.ge('2026-04-17').values
cols = ['n_events', 'item_nunique']

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('доля ботов (это уровень константы):', round(ytr[is_valid].mean(), 4))

## Файл с предсказаниями

In [ ]:
model.fit(Xtr[cols], ytr)
sub = pd.DataFrame({
    'cookie_id': Xte.cookie_id,
    'score': model.predict_proba(Xte[cols])[:, 1],
})
assert len(sub) == len(test) and sub.score.between(0, 1).all()
from pathlib import Path
output_dir = Path('output')
output_dir.mkdir(exist_ok=True)
sub.to_csv(output_dir / 'quickstart_submission.csv', index=False)
sub.head()

## Куда идти дальше

Этот quickstart не пытается описать поведение куки. В `01_eda.ipynb` мы разбираем события и границы окон, затем по шагам сравниваем бейзлайн, CatBoost и новые признаки. Итог и ограничения модели собраны в `13_full_sequence.ipynb`